# 🧠 ICT / SMC Concepts — Testing Notebook
**Author:** Next Level Brain  
**Purpose:** Test all ICT/SMC trading concepts independently from the live bot

---
## 📚 Concepts Covered
| # | Concept | Description |
|---|---------|-------------|
| 1 | MT5 Setup | Connect & Load Data |
| 2 | Market Structure (MSS/BOS/ChoCH) | Trend identification |
| 3 | Liquidity Sweep | Stop Hunt detection |
| 4 | Fair Value Gap (FVG) | Price imbalance zones |
| 5 | Dealing Range | Premium vs Discount |
| 6 | Order Block | Institutional footprint |
| 7 | OTE Levels | Fibonacci 62%-79% |
| 8 | Silver Bullet Windows | High-probability time zones |
| 9 | Liquidity Pool | Take Profit targeting |
| 10 | Full Confluence Engine | Combined signal generator |
| 11 | Walk-Forward Backtest | Historical performance |

> **How to run:** Click each cell and press `Shift+Enter` to execute one by one.

In [21]:
# ═══════════════════════════════════════════════════
# CELL 1 — IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

load_dotenv()

# ─── User Config ─────────────────────────────────────
SYMBOL    = "XAUUSDm"          # Change to your symbol
TIMEFRAME = mt5.TIMEFRAME_M5   # M1, M5, M15, H1 etc.
BARS      = 500                # How many candles to load

print("✅ Imports loaded successfully")
print(f"   Symbol   : {SYMBOL}")
print(f"   Timeframe: {TIMEFRAME}")
print(f"   Bars     : {BARS}")

✅ Imports loaded successfully
   Symbol   : XAUUSDm
   Timeframe: 5
   Bars     : 500


In [22]:
# ═══════════════════════════════════════════════════
# CELL 2 — MT5 CONNECTION & DATA LOADER
# ═══════════════════════════════════════════════════
def connect_mt5():
    terminal_path = r"C:\Program Files\MetaTrader 5 EXNESS\terminal64.exe"
    if not mt5.initialize(path=terminal_path):
        raise RuntimeError(f"MT5 init failed: {mt5.last_error()}")
    login    = int(os.getenv("MT5_LOGIN", "0"))
    password = os.getenv("MT5_PASSWORD", "")
    server   = os.getenv("MT5_SERVER", "")
    if login and password and server:
        if not mt5.login(login, password=password, server=server):
            print(f"⚠️ Login failed: {mt5.last_error()}")
    acc = mt5.account_info()
    if acc:
        print(f"✅ Connected | Account: {acc.login} | Balance: ${acc.balance:.2f}")
    return True

def get_data(symbol=SYMBOL, tf=TIMEFRAME, bars=BARS):
    rates = mt5.copy_rates_from_pos(symbol, tf, 0, bars)
    if rates is None or len(rates) == 0:
        raise ValueError(f"No data returned for {symbol}")
    df = pd.DataFrame(rates)
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df.set_index('time', inplace=True)
    print(f"📊 {len(df)} bars loaded | Latest close: {df['close'].iloc[-1]:.2f} @ {df.index[-1]}")
    return df

connect_mt5()
df = get_data()

✅ Connected | Account: 260150554 | Balance: $2125.18
📊 500 bars loaded | Latest close: 5096.97 @ 2026-03-09 09:00:00


---
## 📐 Concept 1 — Market Structure (MSS / BOS / ChoCH)
ICT uses **market structure** to determine the current trend direction.
- **BOS (Break of Structure)** = A swing high/low is broken, trend continues
- **ChoCH (Change of Character)** = Opposite structure broken, trend reversing
- **MSS (Market Structure Shift)** = First ChoCH after a liquidity sweep

In [23]:
# ═══════════════════════════════════════════════════
# CELL 3 — MARKET STRUCTURE SHIFT (MSS / BOS / ChoCH)
# ═══════════════════════════════════════════════════
def detect_market_structure(df, lookback=20):
    index  = len(df) - 1
    recent = df.iloc[max(0, index-lookback): index+1]
    swing_highs = recent['high'].rolling(3, center=True).max()
    swing_lows  = recent['low'].rolling(3, center=True).min()
    prev_high   = swing_highs.iloc[-2]
    prev_low    = swing_lows.iloc[-2]
    curr_high   = df['high'].iloc[-1]
    curr_low    = df['low'].iloc[-1]

    if   curr_high > prev_high and curr_low > prev_low: structure = "BULLISH_BOS"
    elif curr_low  < prev_low  and curr_high < prev_high: structure = "BEARISH_BOS"
    elif curr_low  < prev_low  and curr_high > prev_high: structure = "CHOCH"
    else: structure = "NEUTRAL"

    emoji = {"BULLISH_BOS": "🟢", "BEARISH_BOS": "🔴", "CHOCH": "🟡", "NEUTRAL": "⚪"}
    print(f"{emoji.get(structure,'⚪')} Market Structure : {structure}")
    print(f"   Prev High : {prev_high:.2f}  |  Curr High : {curr_high:.2f}")
    print(f"   Prev Low  : {prev_low:.2f}  |  Curr Low  : {curr_low:.2f}")
    return {'structure': structure, 'prev_high': prev_high, 'prev_low': prev_low}

mss = detect_market_structure(df)

⚪ Market Structure : NEUTRAL
   Prev High : 5104.10  |  Curr High : 5104.10
   Prev Low  : 5092.36  |  Curr Low  : 5095.20


---
## 💧 Concept 2 — Liquidity Sweep (Stop Hunt)
Banks push price **below swing lows** (to grab sell-stop orders) then reverse UP.  
Or push price **above swing highs** (buy-stops) then reverse DOWN.  
The sweep wick + close back inside = **high-probability reversal signal**.

In [24]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — LIQUIDITY SWEEP DETECTOR v2.2 (Diagnostic & Tunable)
# ═══════════════════════════════════════════════════════════════════════════════
# Features: Auto-tuning, visual sweep markers, detailed diagnostics

import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import List, Optional, Dict, Literal, Tuple
from datetime import datetime

# ───────────────────────────────────────────────────────────────────────────────
# SAFETY CHECK
# ───────────────────────────────────────────────────────────────────────────────

try:
    _ = df.shape
    DATA_SOURCE = "external"
    HAS_VOLUME = 'volume' in df.columns
except NameError:
    print("⚠️  Creating sample data...")
    np.random.seed(42)
    n_bars = 200
    dates = pd.date_range(end=datetime.now(), periods=n_bars, freq='1H')
    
    # Create data WITH deliberate sweep patterns
    price = 5000 + np.cumsum(np.random.normal(0, 2, n_bars))
    
    # Inject sweep patterns
    # Bearish sweep at bar 50
    price[48:52] = [5100, 5120, 5115, 5095]  # High sweep then drop
    # Bullish sweep at bar 120  
    price[118:122] = [4900, 4880, 4885, 4910]  # Low sweep then rise
    
    df = pd.DataFrame({
        'open': price + np.random.normal(0, 1, n_bars),
        'high': price + np.abs(np.random.normal(3, 1, n_bars)),
        'low': price - np.abs(np.random.normal(3, 1, n_bars)),
        'close': price
    }, index=dates)
    
    # Ensure OHLC consistency
    df['high'] = df[['open', 'close', 'high']].max(axis=1)
    df['low'] = df[['open', 'close', 'low']].min(axis=1)
    
    DATA_SOURCE = "generated_with_sweeps"
    HAS_VOLUME = False

# ───────────────────────────────────────────────────────────────────────────────
# DIAGNOSTIC: Analyze your data characteristics first
# ───────────────────────────────────────────────────────────────────────────────

print("═" * 70)
print("🔍 DATA DIAGNOSTICS")
print("═" * 70)

recent = df.tail(50)
avg_range = (recent['high'] - recent['low']).mean()
avg_close = recent['close'].mean()
volatility_pct = (avg_range / avg_close) * 100

print(f"   Asset Price Level : ~{avg_close:.2f}")
print(f"   Avg Bar Range     : {avg_range:.2f} ({volatility_pct:.3f}%)")
print(f"   Typical Wick Size : {(recent['high'] - recent[['open', 'close']].max(axis=1)).mean():.2f}")

# Suggest optimal settings
suggested_pen = max(0.1, volatility_pct * 0.1)
suggested_rej = max(0.05, volatility_pct * 0.05)

print(f"\n💡 Suggested Settings for this volatility:")
print(f"   wick_penetration_min: {suggested_pen:.2f}%")
print(f"   body_rejection_min  : {suggested_rej:.2f}%")

# ───────────────────────────────────────────────────────────────────────────────
# CONFIGURATION (Tunable)
# ───────────────────────────────────────────────────────────────────────────────

SWEEP_CONFIG = {
    'lookback': 20,
    'strength_threshold': 0.3,          # Lowered for more sensitivity
    'wick_penetration_min': suggested_pen,  # Auto-tuned
    'body_rejection_min': suggested_rej,    # Auto-tuned
    'historical_scan_bars': 100,
    'use_volume': HAS_VOLUME,
    'diagnostic_mode': True,            # Extra debug output
}

# ───────────────────────────────────────────────────────────────────────────────
# DATA STRUCTURES
# ───────────────────────────────────────────────────────────────────────────────

@dataclass
class LiquiditySweep:
    type: Literal['BULLISH', 'BEARISH']
    sweep_level: float
    penetration_pct: float
    rejection_pct: float
    strength: float
    bar_idx: int
    timestamp: pd.Timestamp
    volume_ratio: Optional[float] = None
    follow_through: Optional[str] = None
    
    def to_dict(self) -> Dict:
        return {
            'type': self.type,
            'sweep_level': round(self.sweep_level, 2),
            'penetration_pct': round(self.penetration_pct, 4),
            'rejection_pct': round(self.rejection_pct, 4),
            'strength': round(self.strength, 2),
            'bar_idx': self.bar_idx,
            'timestamp': str(self.timestamp),
            'volume_ratio': round(self.volume_ratio, 2) if self.volume_ratio else None,
            'follow_through': self.follow_through
        }


# ───────────────────────────────────────────────────────────────────────────────
# CORE ENGINE
# ───────────────────────────────────────────────────────────────────────────────

class LiquiditySweepDetector:
    
    def __init__(self, df: pd.DataFrame, config: Dict = None):
        self.df = df.copy()
        self.config = {**SWEEP_CONFIG, **(config or {})}
        self.current_idx = len(df) - 1
        self.current_bar = df.iloc[-1]
        self.current_price = float(df['close'].iloc[-1])
        self.has_volume = self.config.get('use_volume', False) and 'volume' in df.columns
        
        if self.has_volume:
            self.avg_volume = df['volume'].rolling(20).mean().iloc[-1]
        else:
            self.avg_volume = 1.0
        
        self.diagnostic_log = []
        
    def _log(self, msg: str):
        if self.config.get('diagnostic_mode'):
            self.diagnostic_log.append(msg)
        
    def _find_swing_levels(self, lookback: int, end_idx: int = None) -> Tuple[float, float, int, int]:
        if end_idx is None:
            end_idx = self.current_idx
            
        start_idx = max(0, end_idx - lookback)
        window = self.df.iloc[start_idx:end_idx]
        
        if window.empty:
            return self.current_price, self.current_price, end_idx, end_idx
            
        swing_low = window['low'].min()
        swing_high = window['high'].max()
        
        # Find indices safely
        low_mask = window['low'] == swing_low
        high_mask = window['high'] == swing_high
        
        low_idx = start_idx + low_mask.values.argmax() if low_mask.any() else start_idx
        high_idx = start_idx + high_mask.values.argmax() if high_mask.any() else start_idx
        
        return swing_low, swing_high, low_idx, high_idx
    
    def _calculate_sweep_metrics(self, level: float, is_high: bool) -> Optional[Dict]:
        bar = self.current_bar
        level_type = "HIGH" if is_high else "LOW"
        
        self._log(f"Testing {level_type} level: {level:.2f}")
        self._log(f"  Bar High: {bar['high']:.2f}, Low: {bar['low']:.2f}, Close: {bar['close']:.2f}")
        
        if is_high:
            if bar['high'] <= level:
                self._log(f"  ❌ High {bar['high']:.2f} not above level {level:.2f}")
                return None
            if bar['close'] >= level:
                self._log(f"  ❌ Close {bar['close']:.2f} not below level {level:.2f}")
                return None
                
            penetration = (bar['high'] - level) / level * 100
            rejection = (level - bar['close']) / level * 100
            
        else:
            if bar['low'] >= level:
                self._log(f"  ❌ Low {bar['low']:.2f} not below level {level:.2f}")
                return None
            if bar['close'] <= level:
                self._log(f"  ❌ Close {bar['close']:.2f} not above level {level:.2f}")
                return None
                
            penetration = (level - bar['low']) / level * 100
            rejection = (bar['close'] - level) / level * 100
        
        self._log(f"  ✅ Penetration: {penetration:.3f}%, Rejection: {rejection:.3f}%")
        
        min_pen = self.config['wick_penetration_min']
        min_rej = self.config['body_rejection_min']
        
        if penetration < min_pen:
            self._log(f"  ❌ Penetration {penetration:.3f}% < min {min_pen:.3f}%")
            return None
            
        if rejection < min_rej:
            self._log(f"  ❌ Rejection {rejection:.3f}% < min {min_rej:.3f}%")
            return None
            
        self._log(f"  ✅ PASSED all thresholds")
        
        return {
            'penetration_pct': penetration,
            'rejection_pct': rejection,
            'level': level
        }
    
    def _calculate_strength(self, metrics: Dict, volume_ratio: float, level_age: int) -> float:
        pen_score = min(metrics['penetration_pct'] / 0.5, 1.0)
        rej_score = min(metrics['rejection_pct'] / 0.3, 1.0)
        age_score = max(1 - (level_age / self.config['lookback']), 0.3)
        
        if self.has_volume and volume_ratio > 0:
            vol_score = min(volume_ratio / 2.0, 1.0)
            strength = (pen_score * 0.35 + rej_score * 0.35 + vol_score * 0.20 + age_score * 0.10)
        else:
            strength = (pen_score * 0.45 + rej_score * 0.40 + age_score * 0.15)
        
        return min(strength, 1.0)
    
    def detect_current_bar(self) -> Optional[LiquiditySweep]:
        self.diagnostic_log = []  # Reset log
        lookback = self.config['lookback']
        
        swing_low, swing_high, low_idx, high_idx = self._find_swing_levels(lookback)
        
        self._log(f"Current Bar {self.current_idx}: Price {self.current_price:.2f}")
        self._log(f"Swing Low: {swing_low:.2f} (bar {low_idx}), High: {swing_high:.2f} (bar {high_idx})")
        
        if self.has_volume:
            current_volume = self.current_bar['volume']
            volume_ratio = current_volume / self.avg_volume if self.avg_volume > 0 else 1.0
        else:
            volume_ratio = None
        
        # Test bullish sweep
        metrics_low = self._calculate_sweep_metrics(swing_low, is_high=False)
        if metrics_low:
            level_age = self.current_idx - low_idx
            strength = self._calculate_strength(metrics_low, volume_ratio or 1.0, level_age)
            self._log(f"Bullish sweep strength: {strength:.2%} (threshold: {self.config['strength_threshold']:.2%})")
            
            if strength >= self.config['strength_threshold']:
                return LiquiditySweep(
                    type='BULLISH',
                    sweep_level=metrics_low['level'],
                    penetration_pct=metrics_low['penetration_pct'],
                    rejection_pct=metrics_low['rejection_pct'],
                    strength=strength,
                    bar_idx=self.current_idx,
                    timestamp=self.df.index[-1],
                    volume_ratio=volume_ratio
                )
        
        # Test bearish sweep
        metrics_high = self._calculate_sweep_metrics(swing_high, is_high=True)
        if metrics_high:
            level_age = self.current_idx - high_idx
            strength = self._calculate_strength(metrics_high, volume_ratio or 1.0, level_age)
            self._log(f"Bearish sweep strength: {strength:.2%} (threshold: {self.config['strength_threshold']:.2%})")
            
            if strength >= self.config['strength_threshold']:
                return LiquiditySweep(
                    type='BEARISH',
                    sweep_level=metrics_high['level'],
                    penetration_pct=metrics_high['penetration_pct'],
                    rejection_pct=metrics_high['rejection_pct'],
                    strength=strength,
                    bar_idx=self.current_idx,
                    timestamp=self.df.index[-1],
                    volume_ratio=volume_ratio
                )
        
        self._log("No sweep detected - failed strength threshold or no pattern")
        return None
    
    def scan_historical(self, bars: int = None) -> List[LiquiditySweep]:
        if bars is None:
            bars = self.config['historical_scan_bars']
            
        sweeps = []
        original_idx = self.current_idx
        
        for i in range(max(10, original_idx - bars), original_idx + 1):
            self.current_idx = i
            self.current_bar = self.df.iloc[i]
            
            sweep = self._detect_at_index(i)
            if sweep:
                sweep.follow_through = self._analyze_follow_through(i, sweep.type)
                sweeps.append(sweep)
        
        self.current_idx = original_idx
        self.current_bar = self.df.iloc[-1]
        
        return sweeps
    
    def _detect_at_index(self, idx: int) -> Optional[LiquiditySweep]:
        lookback = self.config['lookback']
        start = max(0, idx - lookback)
        window = self.df.iloc[start:idx]
        
        if len(window) < 3:
            return None
            
        swing_low = window['low'].min()
        swing_high = window['high'].max()
        
        bar = self.df.iloc[idx]
        
        if self.has_volume:
            avg_vol = self.df.iloc[start:idx]['volume'].mean()
            volume_ratio = bar['volume'] / avg_vol if avg_vol > 0 else 1.0
        else:
            volume_ratio = None
        
        # Bullish sweep
        if bar['low'] < swing_low and bar['close'] > swing_low:
            pen = (swing_low - bar['low']) / swing_low * 100
            rej = (bar['close'] - swing_low) / swing_low * 100
            if pen >= self.config['wick_penetration_min'] and rej >= self.config['body_rejection_min']:
                return LiquiditySweep(
                    type='BULLISH', sweep_level=swing_low,
                    penetration_pct=pen, rejection_pct=rej,
                    strength=min((pen + rej) / 0.8, 1.0),
                    bar_idx=idx, timestamp=self.df.index[idx],
                    volume_ratio=volume_ratio
                )
        
        # Bearish sweep
        if bar['high'] > swing_high and bar['close'] < swing_high:
            pen = (bar['high'] - swing_high) / swing_high * 100
            rej = (swing_high - bar['close']) / swing_high * 100
            if pen >= self.config['wick_penetration_min'] and rej >= self.config['body_rejection_min']:
                return LiquiditySweep(
                    type='BEARISH', sweep_level=swing_high,
                    penetration_pct=pen, rejection_pct=rej,
                    strength=min((pen + rej) / 0.8, 1.0),
                    bar_idx=idx, timestamp=self.df.index[idx],
                    volume_ratio=volume_ratio
                )
        
        return None
    
    def _analyze_follow_through(self, sweep_idx: int, sweep_type: str, check_bars: int = 5) -> str:
        if sweep_idx + check_bars >= len(self.df):
            return 'pending'
            
        subsequent = self.df.iloc[sweep_idx + 1:sweep_idx + check_bars + 1]
        sweep_bar = self.df.iloc[sweep_idx]
        
        if sweep_type == 'BULLISH':
            target = sweep_bar['close'] + (sweep_bar['close'] - sweep_bar['low']) * 1.5
            if (subsequent['high'] >= target).any():
                return 'confirmed'
            elif subsequent['close'].iloc[-1] < sweep_bar['close']:
                return 'failed'
        else:
            target = sweep_bar['close'] - (sweep_bar['high'] - sweep_bar['close']) * 1.5
            if (subsequent['low'] <= target).any():
                return 'confirmed'
            elif subsequent['close'].iloc[-1] > sweep_bar['close']:
                return 'failed'
        
        return 'pending'


# ───────────────────────────────────────────────────────────────────────────────
# DISPLAY FUNCTIONS
# ───────────────────────────────────────────────────────────────────────────────

def print_sweep_details(sweep: LiquiditySweep, detector: LiquiditySweepDetector):
    emoji = "🟢" if sweep.type == 'BULLISH' else "🔴"
    liq_side = "Sell-Side" if sweep.type == 'BULLISH' else "Buy-Side"
    
    print(f"\n{emoji} {sweep.type} LIQUIDITY SWEEP DETECTED")
    print("   " + "─" * 56)
    print(f"   Swept Level     : {sweep.sweep_level:.2f}")
    print(f"   Liquidity Type  : {liq_side} Liquidity")
    print(f"   Wick Penetration: {sweep.penetration_pct:.3f}%")
    print(f"   Body Rejection  : {sweep.rejection_pct:.3f}%")
    print(f"   Strength Score  : {sweep.strength:.1%}")
    
    if detector.has_volume and sweep.volume_ratio:
        print(f"   Volume Ratio    : {sweep.volume_ratio:.2f}x")
    else:
        print(f"   Volume          : N/A")
        
    print("   " + "─" * 56)
    print(f"   ✅ Signal: {sweep.type} reversal likely")

def print_no_sweep(detector: LiquiditySweepDetector, show_diagnostics: bool = True):
    lookback = detector.config['lookback']
    swing_low, swing_high, _, _ = detector._find_swing_levels(lookback)
    
    print(f"\n⚪ NO LIQUIDITY SWEEP on current bar")
    print("   " + "─" * 56)
    print(f"   Current Price   : {detector.current_price:.2f}")
    print(f"   Swing High (20) : {swing_high:.2f}")
    print(f"   Swing Low (20)  : {swing_low:.2f}")
    print(f"   Thresholds      : Pen {detector.config['wick_penetration_min']:.2f}% / Rej {detector.config['body_rejection_min']:.2f}%")
    print("   " + "─" * 56)
    
    if show_diagnostics and detector.diagnostic_log:
        print("   📝 Diagnostic Log:")
        for log in detector.diagnostic_log[-6:]:  # Show last 6 entries
            print(f"      {log}")
        print("   " + "─" * 56)


# ───────────────────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ───────────────────────────────────────────────────────────────────────────────

def run_sweep_analysis(df: pd.DataFrame, config: Dict = None) -> Dict:
    detector = LiquiditySweepDetector(df, config)
    
    current_sweep = detector.detect_current_bar()
    recent_sweeps = detector.scan_historical()
    
    confirmed = len([s for s in recent_sweeps if s.follow_through == 'confirmed'])
    failed = len([s for s in recent_sweeps if s.follow_through == 'failed'])
    total_resolved = confirmed + failed
    success_rate = confirmed / total_resolved if total_resolved > 0 else 0
    
    return {
        'current_sweep': current_sweep,
        'recent_sweeps': recent_sweeps,
        'success_rate': success_rate,
        'detector': detector,
        'has_volume': detector.has_volume,
        'diagnostics': detector.diagnostic_log,
        'config_used': detector.config
    }


# ───────────────────────────────────────────────────────────────────────────────
# RUN ANALYSIS
# ───────────────────────────────────────────────────────────────────────────────

results = run_sweep_analysis(df)

print("\n" + "═" * 70)
print("💧 LIQUIDITY SWEEP ANALYSIS v2.2")
print("═" * 70)

# Current bar
if results['current_sweep']:
    print_sweep_details(results['current_sweep'], results['detector'])
else:
    print_no_sweep(results['detector'], show_diagnostics=True)

# Historical
print(f"\n📊 Historical Sweeps (last {SWEEP_CONFIG['historical_scan_bars']} bars):")
print(f"   Total Found: {len(results['recent_sweeps'])}")
if results['recent_sweeps']:
    confirmed = len([s for s in results['recent_sweeps'] if s.follow_through == 'confirmed'])
    failed = len([s for s in results['recent_sweeps'] if s.follow_through == 'failed'])
    print(f"   Success Rate: {results['success_rate']:.0%} ({confirmed}/{confirmed+failed})")
    
    print(f"\n   Last 5 Sweeps:")
    for s in results['recent_sweeps'][-5:]:
        status = {'confirmed': '✅', 'failed': '❌', 'pending': '⏳'}.get(s.follow_through, '?')
        print(f"      {status} {s.type} @ {s.sweep_level:.2f} | Str: {s.strength:.0%} | {str(s.timestamp)[:16]}")

# Config summary
print(f"\n⚙️  Active Configuration:")
print(f"   Lookback: {results['config_used']['lookback']} bars")
print(f"   Penetration Min: {results['config_used']['wick_penetration_min']:.3f}%")
print(f"   Rejection Min: {results['config_used']['body_rejection_min']:.3f}%")
print(f"   Strength Threshold: {results['config_used']['strength_threshold']:.0%}")

print("═" * 70)

# Exports
sweep = results['current_sweep'].to_dict() if results['current_sweep'] else {'detected': False}
sweep_object = results['current_sweep']
recent_sweeps_list = [s.to_dict() for s in results['recent_sweeps']]

print(f"\n✅ Exports: sweep, sweep_object, recent_sweeps_list")

══════════════════════════════════════════════════════════════════════
🔍 DATA DIAGNOSTICS
══════════════════════════════════════════════════════════════════════
   Asset Price Level : ~5105.18
   Avg Bar Range     : 9.02 (0.177%)
   Typical Wick Size : 2.54

💡 Suggested Settings for this volatility:
   wick_penetration_min: 0.10%
   body_rejection_min  : 0.05%

══════════════════════════════════════════════════════════════════════
💧 LIQUIDITY SWEEP ANALYSIS v2.2
══════════════════════════════════════════════════════════════════════

⚪ NO LIQUIDITY SWEEP on current bar
   ────────────────────────────────────────────────────────
   Current Price   : 5096.97
   Swing High (20) : 5110.60
   Swing Low (20)  : 5086.94
   Thresholds      : Pen 0.10% / Rej 0.05%
   ────────────────────────────────────────────────────────
   📝 Diagnostic Log:
        Bar High: 5104.10, Low: 5095.20, Close: 5096.97
        ❌ Low 5095.20 not below level 5086.94
      Testing HIGH level: 5110.60
        Bar High: 

---
## 📊 Concept 3 — Fair Value Gap (FVG / Imbalance)
FVG = 3-candle pattern where an **imbalance** (gap) exists between candle 1 and candle 3.  
Price tends to **retrace into the gap** before continuing.  
- **Bullish FVG**: Gap between C1.high and C3.low → price came up fast
- **Bearish FVG**: Gap between C1.low and C3.high → price came down fast

In [25]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — FAIR VALUE GAP (FVG) DETECTOR & ACTIVE FVG SCANNER v2.0
# ═══════════════════════════════════════════════════════════════════════════════
# Features: Vectorized operations, ATR-based sizing, fill status tracking, 
#           nearest FVG auto-selection, and robust error handling

import pandas as pd
import numpy as np
from dataclasses import dataclass
from typing import List, Optional, Dict, Literal
from datetime import datetime, timedelta

# ───────────────────────────────────────────────────────────────────────────────
# SAFETY CHECK: Load df if not defined (for standalone testing)
# ───────────────────────────────────────────────────────────────────────────────

try:
    _ = df.shape  # Test if df exists
    DATA_SOURCE = "external"
except NameError:
    print("⚠️  'df' not found. Creating sample OHLC data for testing...")
    
    # Generate realistic sample price data
    np.random.seed(42)
    n_bars = 500
    dates = pd.date_range(end=datetime.now(), periods=n_bars, freq='1h')
    
    # Random walk price generation
    returns = np.random.normal(0.0001, 0.001, n_bars)
    price = 45000 * np.exp(np.cumsum(returns))
    
    # Generate OHLC from close
    noise = np.random.normal(0, 0.0005, n_bars)
    df = pd.DataFrame({
        'open': price * (1 + noise * 0.5),
        'high': price * (1 + np.abs(noise) * 1.5),
        'low': price * (1 - np.abs(noise) * 1.5),
        'close': price,
        'volume': np.random.randint(1000, 10000, n_bars)
    }, index=dates)
    
    # Ensure OHLC consistency
    df['high'] = df[['open', 'close', 'high']].max(axis=1)
    df['low'] = df[['open', 'close', 'low']].min(axis=1)
    
    DATA_SOURCE = "generated"
    print(f"✅ Sample data created: {len(df)} bars from {df.index[0]} to {df.index[-1]}")

# ───────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────────────────────────────────────

FVG_CONFIG = {
    'lookback': 150,                    # Bars to scan for active FVGs
    'min_gap_atr_multiple': 0.15,       # Min gap size as % of ATR (was 0.0005 fixed)
    'atr_period': 14,                   # ATR lookback for dynamic sizing
    'strength_reference_atr': 0.5,      # ATR multiple for 100% strength score
    'max_display_fvgs': 5,              # Top N FVGs to display per type
}

# ───────────────────────────────────────────────────────────────────────────────
# DATA STRUCTURES
# ───────────────────────────────────────────────────────────────────────────────

@dataclass
class FairValueGap:
    type: Literal['BULLISH', 'BEARISH']
    low: float
    high: float
    gap_size: float
    strength: float              # 0.0 to 1.0
    status: Literal['active', 'partial', 'filled']
    bar_idx: int
    timestamp: pd.Timestamp
    distance_pct: float          # Distance from current price
    
    @property
    def mid(self) -> float:
        return (self.low + self.high) / 2
    
    def contains_price(self, price: float) -> bool:
        return self.low <= price <= self.high
    
    def to_dict(self) -> Dict:
        return {
            'type': self.type,
            'low': round(self.low, 2),
            'high': round(self.high, 2),
            'mid': round(self.mid, 2),
            'gap_size': round(self.gap_size, 4),
            'strength': round(self.strength, 2),
            'status': self.status,
            'bar_idx': self.bar_idx,
            'timestamp': str(self.timestamp),
            'distance_pct': round(self.distance_pct, 2),
            'is_inside': False  # Set by scanner
        }


# ───────────────────────────────────────────────────────────────────────────────
# CORE FVG ENGINE
# ───────────────────────────────────────────────────────────────────────────────

class FVGScanner:
    """
    Professional Fair Value Gap detector with ATR-based dynamic sizing
    and three-state fill tracking (active/partial/filled).
    """
    
    def __init__(self, df: pd.DataFrame, config: Dict = None):
        self.df = df.copy()
        self.config = {**FVG_CONFIG, **(config or {})}
        self.current_price = float(df['close'].iloc[-1])
        self.current_idx = len(df) - 1
        
        # Pre-calculate ATR for dynamic gap sizing
        self._calculate_atr()
        
    def _calculate_atr(self):
        """Calculate Average True Range for dynamic gap sizing."""
        period = self.config['atr_period']
        high_low = self.df['high'] - self.df['low']
        high_close = np.abs(self.df['high'] - self.df['close'].shift())
        low_close = np.abs(self.df['low'] - self.df['close'].shift())
        
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        self.df['atr'] = tr.rolling(window=period, min_periods=1).mean()
        
    def _get_fill_status(self, low: float, high: float, created_idx: int) -> str:
        """
        Determine if FVG is: 'active' (untouched), 'partial' (wick only), 
        or 'filled' (close inside).
        """
        if created_idx >= self.current_idx:
            return 'active'
            
        subsequent = self.df.iloc[created_idx + 1:self.current_idx + 1]
        if subsequent.empty:
            return 'active'
        
        # Check if price entered the gap zone (low <= gap_high AND high >= gap_low)
        entered = subsequent[
            (subsequent['low'] <= high) & (subsequent['high'] >= low)
        ]
        
        if entered.empty:
            return 'active'
        
        # Check if any close happened inside the gap (full fill)
        closed_inside = entered[
            (entered['close'] >= low) & (entered['close'] <= high)
        ]
        
        return 'filled' if not closed_inside.empty else 'partial'
    
    def _calculate_strength(self, gap_size: float, atr: float) -> float:
        """Calculate normalized strength score 0.0-1.0."""
        if atr == 0 or pd.isna(atr):
            return 0.5
        reference = self.config['strength_reference_atr']
        return min(gap_size / (atr * reference), 1.0)
    
    def _calculate_distance(self, fvg_type: str, fvg_high: float, fvg_low: float) -> float:
        """Calculate % distance from current price to FVG."""
        if fvg_type == 'BULLISH':
            # For bullish, distance is negative when price is below (needs to rise to fill)
            return (self.current_price - fvg_low) / self.current_price * 100
        else:
            # For bearish, distance is positive when price is below (needs to rise to reach)
            return (fvg_high - self.current_price) / self.current_price * 100
    
    def detect_current_bar(self) -> Optional[FairValueGap]:
        """
        Detect FVG on the most recent 3 bars only (for real-time signals).
        Returns None if no valid FVG detected.
        """
        if len(self.df) < 3:
            return None
            
        b1 = self.df.iloc[-3]
        b2 = self.df.iloc[-2]
        b3 = self.df.iloc[-1]
        atr = self.df['atr'].iloc[-2]
        
        min_gap = atr * self.config['min_gap_atr_multiple']
        
        # Bullish FVG: b1.high < b3.low
        if b1['high'] < b3['low']:
            gap = b3['low'] - b1['high']
            if gap > min_gap:
                strength = self._calculate_strength(gap, atr)
                dist = self._calculate_distance('BULLISH', b3['low'], b1['high'])
                return FairValueGap(
                    type='BULLISH',
                    low=b1['high'],
                    high=b3['low'],
                    gap_size=gap,
                    strength=strength,
                    status='active',  # Current bar is always active
                    bar_idx=self.current_idx,
                    timestamp=self.df.index[-1],
                    distance_pct=dist
                )
        
        # Bearish FVG: b1.low > b3.high
        if b1['low'] > b3['high']:
            gap = b1['low'] - b3['high']
            if gap > min_gap:
                strength = self._calculate_strength(gap, atr)
                dist = self._calculate_distance('BEARISH', b1['low'], b3['high'])
                return FairValueGap(
                    type='BEARISH',
                    low=b3['high'],
                    high=b1['low'],
                    gap_size=gap,
                    strength=strength,
                    status='active',
                    bar_idx=self.current_idx,
                    timestamp=self.df.index[-1],
                    distance_pct=dist
                )
        
        return None
    
    def scan_historical(self) -> List[FairValueGap]:
        """
        Scan historical bars for ALL unfilled/partially filled FVGs.
        Returns list sorted by absolute distance from current price.
        """
        lookback = self.config['lookback']
        start_idx = max(self.config['atr_period'], len(self.df) - lookback)
        
        active_fvgs = []
        
        for i in range(start_idx, len(self.df)):
            b1 = self.df.iloc[i-2]
            b2 = self.df.iloc[i-1]
            b3 = self.df.iloc[i]
            atr = self.df['atr'].iloc[i-1]
            
            if pd.isna(atr) or atr == 0:
                continue
                
            min_gap = atr * self.config['min_gap_atr_multiple']
            
            # Bullish FVG detection
            if b1['high'] < b3['low']:
                gap = b3['low'] - b1['high']
                if gap > min_gap:
                    status = self._get_fill_status(b1['high'], b3['low'], i)
                    if status in ['active', 'partial']:
                        dist = self._calculate_distance('BULLISH', b3['low'], b1['high'])
                        fvg = FairValueGap(
                            type='BULLISH',
                            low=b1['high'],
                            high=b3['low'],
                            gap_size=gap,
                            strength=self._calculate_strength(gap, atr),
                            status=status,
                            bar_idx=i,
                            timestamp=self.df.index[i],
                            distance_pct=dist
                        )
                        active_fvgs.append(fvg)
            
            # Bearish FVG detection
            elif b1['low'] > b3['high']:
                gap = b1['low'] - b3['high']
                if gap > min_gap:
                    status = self._get_fill_status(b3['high'], b1['low'], i)
                    if status in ['active', 'partial']:
                        dist = self._calculate_distance('BEARISH', b1['low'], b3['high'])
                        fvg = FairValueGap(
                            type='BEARISH',
                            low=b3['high'],
                            high=b1['low'],
                            gap_size=gap,
                            strength=self._calculate_strength(gap, atr),
                            status=status,
                            bar_idx=i,
                            timestamp=self.df.index[i],
                            distance_pct=dist
                        )
                        active_fvgs.append(fvg)
        
        # Sort by absolute distance from current price
        return sorted(active_fvgs, key=lambda x: abs(x.distance_pct))
    
    def get_nearest_fvg(self, fvgs: List[FairValueGap], fvg_type: str = None) -> Optional[FairValueGap]:
        """Get nearest FVG to current price, optionally filtered by type."""
        filtered = fvgs
        if fvg_type:
            filtered = [f for f in fvgs if f.type == fvg_type]
        return filtered[0] if filtered else None


# ───────────────────────────────────────────────────────────────────────────────
# DISPLAY & OUTPUT FUNCTIONS
# ───────────────────────────────────────────────────────────────────────────────

def format_fvg_display(fvg: FairValueGap, current_price: float) -> str:
    """Format single FVG for display."""
    inside_tag = "  ⬅ INSIDE!" if fvg.contains_price(current_price) else ""
    status_emoji = {'active': '🟢', 'partial': '🟡', 'filled': '🔴'}[fvg.status]
    
    return (f"      {status_emoji} {fvg.low:.2f} – {fvg.high:.2f}  "
            f"| Gap: {fvg.gap_size:.2f} | Strength: {fvg.strength:.0%} | "
            f"{fvg.distance_pct:+.2f}% from price{inside_tag}")

def print_fvg_section(title: str, fvgs: List[FairValueGap], current_price: float, max_display: int):
    """Print section of FVGs with headers."""
    print(f"\n   {title}: {len(fvgs)} found")
    print("   " + "─" * 56)
    
    if not fvgs:
        print("      None")
        return
        
    for fvg in fvgs[:max_display]:
        print(format_fvg_display(fvg, current_price))
    
    if len(fvgs) > max_display:
        print(f"      ... and {len(fvgs) - max_display} more")


# ───────────────────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ───────────────────────────────────────────────────────────────────────────────

def run_fvg_analysis(df: pd.DataFrame, config: Dict = None) -> Dict:
    """
    Main entry point. Returns dict with current FVG, active FVGs, and nearest selections.
    """
    # Initialize scanner
    scanner = FVGScanner(df, config)
    current_price = scanner.current_price
    
    # Detect current bar FVG
    current_fvg = scanner.detect_current_bar()
    
    # Scan historical active FVGs
    active_fvgs = scanner.scan_historical()
    
    # Separate by type
    bullish_fvgs = [f for f in active_fvgs if f.type == 'BULLISH']
    bearish_fvgs = [f for f in active_fvgs if f.type == 'BEARISH']
    
    # Determine which FVG to use for OTE (Optimal Trade Entry)
    ote_fvg = current_fvg
    ote_source = "current_bar"
    
    if not ote_fvg and active_fvgs:
        ote_fvg = scanner.get_nearest_fvg(active_fvgs)
        ote_source = "nearest_active"
    
    return {
        'current_price': current_price,
        'current_bar_fvg': current_fvg,
        'all_active_fvgs': active_fvgs,
        'bullish_fvgs': bullish_fvgs,
        'bearish_fvgs': bearish_fvgs,
        'ote_fvg': ote_fvg,
        'ote_source': ote_source,
        'scanner': scanner  # For additional queries if needed
    }


# ───────────────────────────────────────────────────────────────────────────────
# RUN ANALYSIS
# ───────────────────────────────────────────────────────────────────────────────

results = run_fvg_analysis(df)

current_price = results['current_price']
current_fvg = results['current_bar_fvg']
active_fvgs = results['all_active_fvgs']
bullish_fvgs = results['bullish_fvgs']
bearish_fvgs = results['bearish_fvgs']
ote_fvg = results['ote_fvg']

# ── Display Results ───────────────────────────────────────────────────────────

print("═" * 70)
print("📊 FAIR VALUE GAP ANALYSIS")
if DATA_SOURCE == "generated":
    print("   [Using generated sample data - replace with your df]")
print("═" * 70)

# Current Bar Status
print(f"\n📍 Current Bar FVG:")
if current_fvg:
    emoji = "🟢" if current_fvg.type == 'BULLISH' else "🔴"
    print(f"   {emoji} {current_fvg.type} @ {current_fvg.low:.2f} – {current_fvg.high:.2f}")
    print(f"      Strength: {current_fvg.strength:.0%} | Gap Size: {current_fvg.gap_size:.4f}")
else:
    print("   ❌ None detected (last 3 bars)")

# Historical Active FVGs
print(f"\n📍 Active FVG Inventory (last {FVG_CONFIG['lookback']} bars):")
print(f"   Total unfilled/partial FVGs: {len(active_fvgs)}")

print_fvg_section(
    f"🟢 Bullish FVGs (Buy magnets below price)", 
    bullish_fvgs, 
    current_price, 
    FVG_CONFIG['max_display_fvgs']
)

print_fvg_section(
    f"🔴 Bearish FVGs (Sell magnets above price)", 
    bearish_fvgs, 
    current_price, 
    FVG_CONFIG['max_display_fvgs']
)

# OTE Selection
print("\n" + "═" * 70)
print("📍 OTE (Optimal Trade Entry) Selection:")
if ote_fvg:
    source_emoji = "⚡" if results['ote_source'] == 'current_bar' else "🎯"
    print(f"   {source_emoji} Selected: {ote_fvg.type} FVG @ {ote_fvg.low:.2f} – {ote_fvg.high:.2f}")
    print(f"      Source: {results['ote_source']} | Distance: {ote_fvg.distance_pct:+.2f}%")
    print(f"      Status: {ote_fvg.status.upper()} | Strength: {ote_fvg.strength:.0%}")
else:
    print("   ❌ No FVG available for OTE")

print("═" * 70)
print(f"   Current Price: {current_price:.2f}")
print("═" * 70)

# ── Export variables for downstream cells ─────────────────────────────────────

# Primary FVG for OTE calculations (guaranteed to exist if any FVG found)
fvg_for_ote = ote_fvg.to_dict() if ote_fvg else {'detected': False}

# All active FVGs as list of dicts
all_fvgs_dict = [f.to_dict() for f in active_fvgs]

# Quick access variables
nearest_bullish = next((f for f in bullish_fvgs if f.status == 'active'), None)
nearest_bearish = next((f for f in bearish_fvgs if f.status == 'active'), None)

print(f"\n✅ Variables exported: fvg_for_ote, all_fvgs_dict, nearest_bullish, nearest_bearish")
print(f"   Data source: {DATA_SOURCE}")

══════════════════════════════════════════════════════════════════════
📊 FAIR VALUE GAP ANALYSIS
══════════════════════════════════════════════════════════════════════

📍 Current Bar FVG:
   ❌ None detected (last 3 bars)

📍 Active FVG Inventory (last 150 bars):
   Total unfilled/partial FVGs: 3

   🟢 Bullish FVGs (Buy magnets below price): 0 found
   ────────────────────────────────────────────────────────
      None

   🔴 Bearish FVGs (Sell magnets above price): 3 found
   ────────────────────────────────────────────────────────
      🟢 5111.04 – 5114.59  | Gap: 3.55 | Strength: 70% | +0.35% from price
      🟢 5118.86 – 5123.78  | Gap: 4.92 | Strength: 96% | +0.53% from price
      🟡 5166.23 – 5170.80  | Gap: 4.56 | Strength: 100% | +1.45% from price

══════════════════════════════════════════════════════════════════════
📍 OTE (Optimal Trade Entry) Selection:
   🎯 Selected: BEARISH FVG @ 5111.04 – 5114.59
      Source: nearest_active | Distance: +0.35%
      Status: ACTIVE | Strength:

---
## 📏 Concept 4 — Dealing Range (Premium vs Discount)
ICT divides the trading range into zones using the **50% equilibrium**:
- **Discount** (below 50%): Ideal zone to **BUY**
- **Premium** (above 50%): Ideal zone to **SELL**
- Buy in Discount + Sell in Premium = trade with institutional flow

In [26]:
# ═══════════════════════════════════════════════════
# CELL 6 — DEALING RANGE (PREMIUM / DISCOUNT)
# ═══════════════════════════════════════════════════
def analyze_dealing_range(df, lookback=20):
    index   = len(df) - 1
    recent  = df.iloc[max(0, index-lookback): index+1]
    rh      = float(recent['high'].max())
    rl      = float(recent['low'].min())
    current = float(df['close'].iloc[-1])
    rng     = rh - rl

    eq   = rl + rng * 0.50
    l25  = rl + rng * 0.25
    l75  = rl + rng * 0.75
    pct  = (current - rl) / rng * 100 if rng > 0 else 50
    zone = 'DISCOUNT 🟢 (BUY zone)' if current < eq else 'PREMIUM 🔴 (SELL zone)'

    print(f"📏 Dealing Range  (last {lookback} bars)")
    print(f"   Range High     : {rh:.2f}")
    print(f"   75% Level      : {l75:.2f}")
    print(f"   50% Equilibrium: {eq:.2f}  ← (Fair Value)")
    print(f"   25% Level      : {l25:.2f}")
    print(f"   Range Low      : {rl:.2f}")
    print(f"   ─────────────────────────")
    print(f"   Current Price  : {current:.2f}  ({pct:.1f}% in range)")
    print(f"   Zone           : {zone}")
    return {'zone': 'DISCOUNT' if current < eq else 'PREMIUM', 'eq': eq, 'l25': l25, 'l75': l75, 'rh': rh, 'rl': rl}

dealing = analyze_dealing_range(df)

📏 Dealing Range  (last 20 bars)
   Range High     : 5110.60
   75% Level      : 5104.69
   50% Equilibrium: 5098.77  ← (Fair Value)
   25% Level      : 5092.85
   Range Low      : 5086.94
   ─────────────────────────
   Current Price  : 5096.97  (42.4% in range)
   Zone           : DISCOUNT 🟢 (BUY zone)


---
## 🏦 Concept 5 — Order Block (Institutional Footprint)
An **Order Block** is the last opposing candle before a significant move.  
- **Bullish OB**: Last bearish candle before a big bullish move
- **Bearish OB**: Last bullish candle before a big bearish move

Institutions leave unfilled orders here — price often returns to this zone.

In [27]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — ORDER BLOCK DETECTOR v2.0
# ═══════════════════════════════════════════════════════════════════════════════
# Features: Class-based architecture, multi-criteria validation, 
#           mitigation tracking, and confluence scoring

import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Literal, Tuple
from datetime import datetime

# ───────────────────────────────────────────────────────────────────────────────
# SAFETY CHECK
# ───────────────────────────────────────────────────────────────────────────────

try:
    _ = df.shape
    DATA_SOURCE = "external"
    HAS_MSS = 'mss' in globals()
except NameError:
    print("⚠️  Creating sample data...")
    np.random.seed(42)
    n_bars = 300
    dates = pd.date_range(end=datetime.now(), periods=n_bars, freq='30min')
    
    # Create data with clear order block patterns
    trend = np.cumsum(np.random.normal(0.0002, 0.001, n_bars))
    price = 50000 * np.exp(trend)
    
    # Inject bullish order block pattern at bar 100
    price[98:103] = [49500, 49400, 49600, 49800, 50100]  # Drop then strong rise
    
    df = pd.DataFrame({
        'open': price * (1 + np.random.normal(0, 0.0002, n_bars)),
        'high': price * (1 + np.abs(np.random.normal(0.0005, 0.0003, n_bars))),
        'low': price * (1 - np.abs(np.random.normal(0.0005, 0.0003, n_bars))),
        'close': price
    }, index=dates)
    
    df['high'] = df[['open', 'close', 'high']].max(axis=1)
    df['low'] = df[['open', 'close', 'low']].min(axis=1)
    
    # Mock MSS for testing
    mss = {'structure': 'BULLISH', 'strength': 0.8}
    HAS_MSS = True
    DATA_SOURCE = "generated"

# ───────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ───────────────────────────────────────────────────────────────────────────────

OB_CONFIG = {
    'lookback':100,                     # Bars to scan for order blocks
    'min_impulse_pct': 0.003,           # Minimum 0.3% move to qualify as impulse
    'max_ob_width_pct': 0.015,          # Maximum OB width as % of price
    'min_bars_since_ob': 3,             # Minimum bars since OB formation
    'mitigation_lookforward': 10,       # Bars to check for mitigation
    'use_volume': 'volume' in df.columns,
    'strength_factors': {
        'impulse_size': 0.35,           # Weight of initial impulse
        'volume_confirmation': 0.25,    # Weight of volume (if available)
        'freshness': 0.20,              # Weight of how recent/unmitigated
        'alignment': 0.20               # Weight of alignment with higher timeframe
    }
}

# ───────────────────────────────────────────────────────────────────────────────
# DATA STRUCTURES
# ───────────────────────────────────────────────────────────────────────────────

@dataclass
class OrderBlock:
    type: Literal['BULLISH', 'BEARISH']
    high: float
    low: float
    open_price: float
    close_price: float
    impulse_pct: float
    strength: float
    bar_idx: int
    timestamp: pd.Timestamp
    volume_ratio: Optional[float] = None
    status: Literal['fresh', 'mitigated', 'partial'] = 'fresh'
    mitigation_bar: Optional[int] = None
    
    @property
    def mid(self) -> float:
        return (self.high + self.low) / 2
    
    @property
    def height(self) -> float:
        return self.high - self.low
    
    def contains_price(self, price: float) -> bool:
        return self.low <= price <= self.high
    
    def to_dict(self) -> Dict:
        return {
            'type': self.type,
            'high': round(self.high, 2),
            'low': round(self.low, 2),
            'mid': round(self.mid, 2),
            'height': round(self.height, 2),
            'impulse_pct': round(self.impulse_pct, 4),
            'strength': round(self.strength, 2),
            'bar_idx': self.bar_idx,
            'timestamp': str(self.timestamp),
            'volume_ratio': round(self.volume_ratio, 2) if self.volume_ratio else None,
            'status': self.status,
            'mitigation_bar': self.mitigation_bar,
            'detected': True
        }


# ───────────────────────────────────────────────────────────────────────────────
# CORE ORDER BLOCK ENGINE
# ───────────────────────────────────────────────────────────────────────────────

class OrderBlockDetector:
    """
    Professional Order Block detector following ICT/SMC methodology.
    Identifies institutional order blocks with impulse validation.
    """
    
    def __init__(self, df: pd.DataFrame, config: Dict = None):
        self.df = df.copy()
        self.config = {**OB_CONFIG, **(config or {})}
        self.current_idx = len(df) - 1
        self.current_price = float(df['close'].iloc[-1])
        self.has_volume = self.config['use_volume'] and 'volume' in df.columns
        
        if self.has_volume:
            self.avg_volume = df['volume'].rolling(20).mean()
        else:
            self.avg_volume = pd.Series(1.0, index=df.index)
    
    def _is_impulse_bar(self, bar_idx: int, direction: str) -> Tuple[bool, float]:
        """
        Check if bar at index was an impulse bar (strong move).
        Returns (is_impulse, impulse_size_pct).
        """
        if bar_idx >= len(self.df) - 1:
            return False, 0.0
            
        bar = self.df.iloc[bar_idx]
        next_bar = self.df.iloc[bar_idx + 1]
        
        price_change = next_bar['close'] - bar['close']
        impulse_pct = abs(price_change) / bar['close']
        
        if direction == 'BULLISH':
            is_impulse = price_change > 0 and impulse_pct >= self.config['min_impulse_pct']
        else:
            is_impulse = price_change < 0 and impulse_pct >= self.config['min_impulse_pct']
            
        return is_impulse, impulse_pct if is_impulse else 0.0
    
    def _validate_ob_criteria(self, bar_idx: int, direction: str) -> Optional[Dict]:
        """
        Validate if bar qualifies as order block foundation.
        Checks: impulse strength, OB width, volume (if available).
        """
        bar = self.df.iloc[bar_idx]
        next_bar = self.df.iloc[bar_idx + 1]
        
        is_impulse, impulse_pct = self._is_impulse_bar(bar_idx, direction)
        if not is_impulse:
            return None
        
        # Calculate OB zone (the bar before impulse)
        ob_high = bar['high']
        ob_low = bar['low']
        ob_width_pct = (ob_high - ob_low) / bar['close']
        
        # OB shouldn't be too wide (indicates chop, not institutional)
        if ob_width_pct > self.config['max_ob_width_pct']:
            return None
        
        # Volume confirmation (optional)
        volume_ratio = None
        if self.has_volume:
            vol = bar['volume']
            avg_vol = self.avg_volume.iloc[bar_idx]
            volume_ratio = vol / avg_vol if avg_vol > 0 else 1.0
            # Require above average volume for strong OB
            if volume_ratio < 0.8:
                return None
        
        return {
            'bar_idx': bar_idx,
            'high': ob_high,
            'low': ob_low,
            'open_price': bar['open'],
            'close_price': bar['close'],
            'impulse_pct': impulse_pct,
            'volume_ratio': volume_ratio,
            'width_pct': ob_width_pct
        }
    
    def _calculate_strength(self, ob_data: Dict, market_bias: str = None) -> float:
        """
        Calculate composite strength score for order block.
        """
        factors = self.config['strength_factors']
        
        # Impulse size score (normalized to 0-1)
        impulse_score = min(ob_data['impulse_pct'] / (self.config['min_impulse_pct'] * 3), 1.0)
        
        # Volume score
        volume_score = 0.5
        if ob_data.get('volume_ratio'):
            volume_score = min(ob_data['volume_ratio'] / 2.0, 1.0)
        
        # Freshness (age since formation)
        bars_since = self.current_idx - ob_data['bar_idx']
        freshness_score = max(1 - (bars_since / self.config['lookback']), 0.3)
        
        # Bias alignment
        alignment_score = 0.5
        if market_bias:
            if market_bias == ob_data['direction']:
                alignment_score = 1.0
            else:
                alignment_score = 0.2
        
        # Weighted composite
        strength = (
            impulse_score * factors['impulse_size'] +
            volume_score * factors['volume_confirmation'] +
            freshness_score * factors['freshness'] +
            alignment_score * factors['alignment']
        )
        
        return min(strength, 1.0)
    
    def _check_mitigation(self, ob: OrderBlock) -> str:
        """
        Check if order block has been mitigated (price returned to zone).
        """
        if ob.bar_idx >= self.current_idx:
            return 'fresh'
            
        subsequent = self.df.iloc[ob.bar_idx + 1:min(ob.bar_idx + self.config['mitigation_lookforward'] + 1, len(self.df))]
        if subsequent.empty:
            return 'fresh'
        
        # Check if price re-entered OB zone
        for i, (_, bar) in enumerate(subsequent.iterrows()):
            if ob.contains_price(bar['close']):
                return 'mitigated'
            # Partial mitigation (wick touched)
            if bar['low'] <= ob.high and bar['high'] >= ob.low:
                return 'partial'
        
        return 'fresh'
    
    def detect_all_blocks(self, direction: str = None, min_strength: float = 0.0) -> List[OrderBlock]:
        """
        Scan for all valid order blocks in lookback period.
        """
        lookback = self.config['lookback']
        start_idx = max(0, self.current_idx - lookback)
        
        blocks = []
        
        for i in range(start_idx, self.current_idx - 1):
            # Skip if too recent (need time to prove it's an OB)
            if self.current_idx - i < self.config['min_bars_since_ob']:
                continue
            
            # Determine direction if not specified
            check_directions = [direction] if direction else ['BULLISH', 'BEARISH']
            
            for dir in check_directions:
                ob_data = self._validate_ob_criteria(i, dir)
                if not ob_data:
                    continue
                
                # Create OrderBlock object
                ob = OrderBlock(
                    type=dir,
                    high=ob_data['high'],
                    low=ob_data['low'],
                    open_price=ob_data['open_price'],
                    close_price=ob_data['close_price'],
                    impulse_pct=ob_data['impulse_pct'],
                    strength=0.0,  # Will calculate next
                    bar_idx=ob_data['bar_idx'],
                    timestamp=self.df.index[ob_data['bar_idx']],
                    volume_ratio=ob_data.get('volume_ratio')
                )
                
                # Calculate strength with bias alignment
                ob.strength = self._calculate_strength({**ob_data, 'direction': dir}, direction)
                
                # Check mitigation status
                ob.status = self._check_mitigation(ob)
                
                if ob.strength >= min_strength:
                    blocks.append(ob)
        
        # Sort by strength descending
        return sorted(blocks, key=lambda x: x.strength, reverse=True)
    
    def get_nearest_block(self, direction: str = None, price: float = None, 
                         require_fresh: bool = True) -> Optional[OrderBlock]:
        """
        Get the nearest relevant order block to current price.
        """
        if price is None:
            price = self.current_price
            
        blocks = self.detect_all_blocks(direction)
        
        if require_fresh:
            blocks = [b for b in blocks if b.status == 'fresh']
        
        if not blocks:
            return None
        
        # Find nearest by distance to OB zone
        def distance_to_ob(ob: OrderBlock) -> float:
            if ob.contains_price(price):
                return 0
            return min(abs(price - ob.high), abs(price - ob.low))
        
        return min(blocks, key=distance_to_ob)
    
    def get_last_block_by_bias(self, market_bias: str) -> Optional[OrderBlock]:
        """
        Get the most recent order block aligned with market bias.
        """
        # For bullish bias, we want bullish OBs (support)
        # For bearish bias, we want bearish OBs (resistance)
        target_type = market_bias
        
        blocks = self.detect_all_blocks(target_type, min_strength=0.4)
        fresh_blocks = [b for b in blocks if b.status == 'fresh']
        
        return fresh_blocks[0] if fresh_blocks else (blocks[0] if blocks else None)


# ───────────────────────────────────────────────────────────────────────────────
# DISPLAY FUNCTIONS
# ───────────────────────────────────────────────────────────────────────────────

def print_order_block(ob: OrderBlock, detector: OrderBlockDetector):
    """Pretty print order block details."""
    emoji = "🟢" if ob.type == 'BULLISH' else "🔴"
    status_emoji = {'fresh': '✨', 'partial': '⚠️', 'mitigated': '❌'}[ob.status]
    
    print(f"\n{emoji} {ob.type} ORDER BLOCK FOUND")
    print("   " + "─" * 56)
    print(f"   Zone          : {ob.low:.2f} – {ob.high:.2f} (mid: {ob.mid:.2f})")
    print(f"   Status        : {status_emoji} {ob.status.upper()}")
    print(f"   Impulse Size  : {ob.impulse_pct:.2%}")
    print(f"   Strength      : {ob.strength:.0%}")
    
    if detector.has_volume and ob.volume_ratio:
        print(f"   Volume Ratio  : {ob.volume_ratio:.2f}x avg")
    
    print(f"   Formed        : {str(ob.timestamp)[:16]} (bar {ob.bar_idx})")
    print("   " + "─" * 56)
    
    if ob.status == 'fresh':
        print(f"   ✅ TRADE SETUP: Enter on return to OB zone")
        print(f"      Stop Loss: Beyond {ob.low if ob.type == 'BULLISH' else ob.high:.2f}")
    elif ob.status == 'partial':
        print(f"   ⚠️  CAUTION: Partially mitigated, reduced confidence")
    else:
        print(f"   ❌ INVALID: Block already mitigated")


# ───────────────────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ───────────────────────────────────────────────────────────────────────────────

def run_order_block_analysis(df: pd.DataFrame, bias: str = None, 
                              mss_data: Dict = None, config: Dict = None) -> Dict:
    """
    Main entry point for order block analysis.
    """
    detector = OrderBlockDetector(df, config)
    
    # Auto-detect bias if not provided
    if bias is None and mss_data:
        bias = 'BULLISH' if 'BULLISH' in mss_data.get('structure', '') else 'BEARISH'
    elif bias is None:
        bias = 'BULLISH'  # Default
    
    # Get primary block aligned with bias
    primary_ob = detector.get_last_block_by_bias(bias)
    
    # Also get all blocks for context
    all_blocks = detector.detect_all_blocks(min_strength=0.3)
    
    # Find nearest block regardless of type
    nearest = detector.get_nearest_block(require_fresh=True)
    
    return {
        'primary_ob': primary_ob,
        'nearest_ob': nearest,
        'all_blocks': all_blocks,
        'bias_used': bias,
        'detector': detector,
        'bullish_count': len([b for b in all_blocks if b.type == 'BULLISH']),
        'bearish_count': len([b for b in all_blocks if b.type == 'BEARISH'])
    }


# ───────────────────────────────────────────────────────────────────────────────
# RUN ANALYSIS
# ───────────────────────────────────────────────────────────────────────────────

# Determine bias from MSS or default
if HAS_MSS:
    bias_guess = 'BULLISH' if 'BULLISH' in mss.get('structure', '') else 'BEARISH'
else:
    bias_guess = 'BULLISH'

results = run_order_block_analysis(df, bias=bias_guess, mss_data=mss if HAS_MSS else None)

print("═" * 70)
print("🏦 ORDER BLOCK ANALYSIS v2.0")
if DATA_SOURCE == "generated":
    print("   [Using generated sample data]")
print("═" * 70)

# Display primary result
if results['primary_ob']:
    print_order_block(results['primary_ob'], results['detector'])
else:
    print_no_blocks(results['detector'], results['bias_used'])

# Inventory
print(f"\n📊 Order Block Inventory:")
print(f"   Total Found     : {len(results['all_blocks'])}")
print(f"   Bullish (Support): {results['bullish_count']}")
print(f"   Bearish (Resist) : {results['bearish_count']}")

if results['all_blocks']:
    print(f"\n   Top 3 by Strength:")
    for ob in results['all_blocks'][:3]:
        status_icon = {'fresh': '✨', 'partial': '⚠️', 'mitigated': '❌'}[ob.status]
        dist = (ob.mid - results['detector'].current_price) / results['detector'].current_price * 100
        print(f"      {status_icon} {ob.type} @ {ob.mid:.2f} | Str: {ob.strength:.0%} | {dist:+.2f}% away")

# Nearest alternative
if results['nearest_ob'] and results['nearest_ob'] != results['primary_ob']:
    print(f"\n🎯 Nearest Alternative:")
    print(f"   {results['nearest_ob'].type} @ {results['nearest_ob'].low:.2f}–{results['nearest_ob'].high:.2f}")

print("═" * 70)

# ── Export variables ───────────────────────────────────────────────────────────

ob = results['primary_ob'].to_dict() if results['primary_ob'] else {'detected': False}
ob_object = results['primary_ob']
all_obs_list = [b.to_dict() for b in results['all_blocks']]
nearest_ob_dict = results['nearest_ob'].to_dict() if results['nearest_ob'] else None

print(f"\n✅ Exports: ob (dict), ob_object (class), all_obs_list, nearest_ob_dict")
print(f"   Bias: {results['bias_used']} | Data: {DATA_SOURCE}")

══════════════════════════════════════════════════════════════════════
🏦 ORDER BLOCK ANALYSIS v2.0
══════════════════════════════════════════════════════════════════════

🔴 BEARISH ORDER BLOCK FOUND
   ────────────────────────────────────────────────────────
   Zone          : 5039.05 – 5050.53 (mid: 5044.79)
   Status        : ⚠️ PARTIAL
   Impulse Size  : 0.31%
   Strength      : 51%
   Formed        : 2026-03-09 00:55 (bar 402)
   ────────────────────────────────────────────────────────
   ⚠️  CAUTION: Partially mitigated, reduced confidence

📊 Order Block Inventory:
   Total Found     : 6
   Bullish (Support): 4
   Bearish (Resist) : 2

   Top 3 by Strength:
      ⚠️ BULLISH @ 5113.43 | Str: 47% | +0.32% away
      ⚠️ BULLISH @ 5045.66 | Str: 44% | -1.01% away
      ⚠️ BULLISH @ 5032.09 | Str: 42% | -1.27% away
══════════════════════════════════════════════════════════════════════

✅ Exports: ob (dict), ob_object (class), all_obs_list, nearest_ob_dict
   Bias: BEARISH | Data: exter

---
## 🎯 Concept 6 — OTE (Optimal Trade Entry) — Fibonacci 62%-79%
ICT's **OTE Zone** = the 62% to 79% Fibonacci retracement of the FVG range.  
When price retraces INTO this zone after a sweep + FVG = **highest probability entry.**

In [28]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 — OTE (OPTIMAL TRADE ENTRY) & FIBONACCI ZONES
# ═══════════════════════════════════════════════════════════════════════════════
# Features: Calculates ICT OTE (62%, 70.5%, 79%) based on FVGs or swings,
#           identifies Consequent Encroachment (50%), and checks if price is in zone.

import pandas as pd
import numpy as np

# ───────────────────────────────────────────────────────────────────────────────
# SAFETY CHECK & MOCK DATA
# ───────────────────────────────────────────────────────────────────────────────

try:
    _ = df.shape
except NameError:
    print("⚠️  Creating sample data for OTE...")
    dates = pd.date_range(end=pd.Timestamp.now(), periods=100, freq='15min')
    df = pd.DataFrame({'close': np.random.normal(5100, 10, 100)}, index=dates)

# Check if we have an FVG from previous cells, otherwise mock it
if 'fvg' not in globals() and 'fvg_dict' not in globals():
    print("⚠️  No active FVG found from previous cells. Creating mock FVG...")
    fvg = {
        'detected': True,
        'type': 'BULLISH',
        'high': float(df['close'].iloc[-1]) + 20.0,
        'low': float(df['close'].iloc[-1]) - 10.0
    }
else:
    # Fallback to existing fvg variable (supports both fvg and fvg_dict naming)
    fvg = fvg if 'fvg' in globals() else fvg_dict

if 'bias_guess' not in globals():
    bias_guess = 'BULLISH'

# ───────────────────────────────────────────────────────────────────────────────
# CORE OTE ENGINE
# ───────────────────────────────────────────────────────────────────────────────

def check_ote_levels(df: pd.DataFrame, target_zone: dict, zone_type: str = "FVG", bias: str = "BULLISH") -> dict:
    """
    Calculates Optimal Trade Entry (OTE) levels for a given price zone (FVG or Swing).
    Standard ICT OTE levels: 62%, 70.5% (Sweet Spot), and 79%.
    """
    if not target_zone or not target_zone.get('detected', True):
        print(f"🎯 OTE: Requires an active {zone_type} — none detected.")
        return {'valid': False}
        
    current = float(df['close'].iloc[-1])
    high = target_zone['high']
    low = target_zone['low']
    zone_range = high - low
    
    # Calculate levels based on bias (retracement direction)
    if bias == 'BULLISH':
        # Price pulls BACK DOWN into the zone. Measuring from High to Low.
        ote_62 = high - (zone_range * 0.62)
        ote_70 = high - (zone_range * 0.705)  # 70.5% is the ICT Sweet Spot
        ote_79 = high - (zone_range * 0.79)
        ce_50  = high - (zone_range * 0.50)   # Consequent Encroachment / Midpoint
        
        in_ote = ote_79 <= current <= ote_62
        distance_to_sweetspot = ((current - ote_70) / ote_70) * 100
    else:
        # Price pulls BACK UP into the zone. Measuring from Low to High.
        ote_62 = low + (zone_range * 0.62)
        ote_70 = low + (zone_range * 0.705)
        ote_79 = low + (zone_range * 0.79)
        ce_50  = low + (zone_range * 0.50)
        
        in_ote = ote_62 <= current <= ote_79
        distance_to_sweetspot = ((current - ote_70) / ote_70) * 100

    # Output formatting
    emoji = "🟢" if bias == 'BULLISH' else "🔴"
    print(f"\n{emoji} {bias} OTE FIBONACCI LEVELS ({zone_type} {low:.2f} - {high:.2f})")
    print("   " + "─" * 56)
    print(f"   0.0% (Entry) : {high if bias=='BULLISH' else low:.2f}  ← Zone Boundary")
    print(f"   50.0% (CE)   : {ce_50:.2f}  ← Consequent Encroachment (EQ)")
    print(f"   62.0% Level  : {ote_62:.2f}  ← OTE Top")
    print(f"   70.5% Level  : {ote_70:.2f}  ← OTE Sweet Spot 🎯")
    print(f"   79.0% Level  : {ote_79:.2f}  ← OTE Bottom")
    print(f"   100% (End)   : {low if bias=='BULLISH' else high:.2f}  ← Zone Invalidation")
    print("   " + "─" * 56)
    print(f"   Current Price: {current:.2f} ({abs(distance_to_sweetspot):.2f}% away from Sweet Spot)")
    
    # Status determination
    if in_ote:
        print(f"   Status       : ✅ YES — Price is in the Prime Entry Zone!")
    else:
        if (bias == 'BULLISH' and current > ote_62) or (bias == 'BEARISH' and current < ote_62):
            print(f"   Status       : ⏳ Waiting — Price hasn't retraced to OTE yet")
        else:
            print(f"   Status       : ⚠️ Past OTE — Price broke through optimal entry")

    return {
        'valid': in_ote,
        'ce_50': ce_50,
        'ote_62': ote_62,
        'ote_70': ote_70,
        'ote_79': ote_79,
        'distance_to_sweet_spot_pct': distance_to_sweetspot
    }


# ───────────────────────────────────────────────────────────────────────────────
# RUN ANALYSIS
# ───────────────────────────────────────────────────────────────────────────────

print("═" * 70)
print("🎯 OTE (OPTIMAL TRADE ENTRY) ANALYSIS v2.0")
print("═" * 70)

# Detect FVG bias or fallback to global bias
fvg_bias = fvg.get('type', bias_guess) if type(fvg) == dict else bias_guess

# Run OTE calculation
ote_results = check_ote_levels(df, target_zone=fvg, zone_type="FVG", bias=fvg_bias)

print("═" * 70)

# ── Export variables ───────────────────────────────────────────────────────────
ote_dict = ote_results
print(f"\n✅ Exports: ote_dict")

══════════════════════════════════════════════════════════════════════
🎯 OTE (OPTIMAL TRADE ENTRY) ANALYSIS v2.0
══════════════════════════════════════════════════════════════════════

🟢 BULLISH OTE FIBONACCI LEVELS (FVG 5085.28 - 5115.28)
   ────────────────────────────────────────────────────────
   0.0% (Entry) : 5115.28  ← Zone Boundary
   50.0% (CE)   : 5100.28  ← Consequent Encroachment (EQ)
   62.0% Level  : 5096.68  ← OTE Top
   70.5% Level  : 5094.13  ← OTE Sweet Spot 🎯
   79.0% Level  : 5091.58  ← OTE Bottom
   100% (End)   : 5085.28  ← Zone Invalidation
   ────────────────────────────────────────────────────────
   Current Price: 5096.97 (0.06% away from Sweet Spot)
   Status       : ⏳ Waiting — Price hasn't retraced to OTE yet
══════════════════════════════════════════════════════════════════════

✅ Exports: ote_dict


---
## ⏰ Concept 7 — Silver Bullet Time Windows
ICT's **Silver Bullet** strategy only trades during 3 specific 1-hour windows (New York time):
- **3 AM – 4 AM** → London Open (most volatile)
- **10 AM – 11 AM** → New York AM Session Open
- **2 PM – 3 PM** → New York PM Session

Trading ONLY during these windows = higher win rate.

In [29]:
# ═══════════════════════════════════════════════════
# CELL 9 — SILVER BULLET TIME WINDOWS
# ═══════════════════════════════════════════════════
def check_silver_bullet_time(symbol=SYMBOL):
    tick = mt5.symbol_info_tick(symbol)
    server_time = datetime.fromtimestamp(tick.time) if tick else datetime.utcnow()
    offset = int(os.getenv("MT5_SERVER_TIME_OFFSET", 0))
    adj    = server_time + timedelta(hours=offset)
    h      = adj.hour

    windows = {3: "London Open", 10: "NY AM Session", 14: "NY PM Session"}
    is_sb   = h in windows
    name    = windows.get(h, "Outside Windows")

    all_windows = [(3,4,'London Open'), (10,11,'NY AM Session'), (14,15,'NY PM Session')]
    
    print(f"⏰ Silver Bullet Time Analysis")
    print(f"   Server Time  : {server_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Adjusted Hour: {h:02d}:00")
    print(f"   ─────────────────────────")
    for s, e, n in all_windows:
        active = '✅ ACTIVE NOW' if h == s else '   '
        print(f"   {active}  {s:02d}:00 - {e:02d}:00  →  {n}")
    print(f"   ─────────────────────────")
    print(f"   Status: {'🎯 PRIME ENTRY WINDOW — Trade aggresively!' if is_sb else '⏳ Wait for Silver Bullet window'}")
    return {'active': is_sb, 'hour': h, 'window': name}

sb = check_silver_bullet_time()

⏰ Silver Bullet Time Analysis
   Server Time  : 2026-03-09 14:04:19
   Adjusted Hour: 14:00
   ─────────────────────────
        03:00 - 04:00  →  London Open
        10:00 - 11:00  →  NY AM Session
   ✅ ACTIVE NOW  14:00 - 15:00  →  NY PM Session
   ─────────────────────────
   Status: 🎯 PRIME ENTRY WINDOW — Trade aggresively!


---
## 🎯 Concept 8 — Liquidity Pool (Take Profit Target)
ICT says banks always target the **next pool of liquidity** — clusters of stop-loss orders sitting above swing highs or below swing lows.  
Use this as your **Take Profit target** — price is magnetically drawn there.

In [30]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10 — LIQUIDITY POOLS (TAKE PROFIT TARGETING)
# ═══════════════════════════════════════════════════════════════════════════════
# Features: Swing point detection, Buy-Side (BSL) & Sell-Side (SSL) liquidity 
#           identification, and multi-tier Take Profit targeting.

import pandas as pd
import numpy as np

# ───────────────────────────────────────────────────────────────────────────────
# SAFETY CHECK & MOCK DATA
# ───────────────────────────────────────────────────────────────────────────────

try:
    _ = df.shape
except NameError:
    print("⚠️  Creating sample data for Liquidity Pools...")
    dates = pd.date_range(end=pd.Timestamp.now(), periods=100, freq='15min')
    df = pd.DataFrame({
        'high': np.random.normal(5150, 20, 100),
        'low': np.random.normal(5050, 20, 100),
        'close': np.random.normal(5100, 15, 100)
    }, index=dates)

# Fallback for Market Structure Shift (MSS) bias if not run previously
if 'mss' not in globals():
    print("⚠️  No active Market Structure Shift (MSS) found. Defaulting to BULLISH bias...")
    mss = {'structure': 'BULLISH', 'strength': 0.8}

# ───────────────────────────────────────────────────────────────────────────────
# CORE LIQUIDITY ENGINE
# ───────────────────────────────────────────────────────────────────────────────

def find_liquidity_pools(df: pd.DataFrame, bias: str = 'BULLISH', lookback: int = 50, swing_window: int = 3) -> dict:
    """
    Identifies major and minor liquidity pools (Take Profit targets) by finding 
    recent swing highs (for BSL) or swing lows (for SSL).
    """
    current_price = float(df['close'].iloc[-1])
    recent_df = df.iloc[max(0, len(df) - lookback): -1].copy() # Exclude current forming bar
    
    if recent_df.empty:
        return {'valid': False}

    # Identify true swing points (peaks and valleys)
    recent_df['is_swing_high'] = recent_df['high'] == recent_df['high'].rolling(window=swing_window*2+1, center=True).max()
    recent_df['is_swing_low'] = recent_df['low'] == recent_df['low'].rolling(window=swing_window*2+1, center=True).min()

    if bias == 'BULLISH':
        # We target Buy-Side Liquidity (BSL) resting above old highs
        target_type = "Buy-Side Liquidity (BSL)"
        target_desc = "above swing highs"
        emoji = "📈"
        
        # Find all swing highs above current price
        valid_targets = recent_df[(recent_df['is_swing_high']) & (recent_df['high'] > current_price)]['high']
        
        if not valid_targets.empty:
            nearest_tp = float(valid_targets.min())
            major_tp = float(valid_targets.max())
        else:
            # Fallback if no prominent swings found
            highs_above = recent_df['high'][recent_df['high'] > current_price]
            nearest_tp = float(highs_above.min()) if not highs_above.empty else current_price * 1.01
            major_tp = float(highs_above.max()) if not highs_above.empty else current_price * 1.02
            
    else:
        # We target Sell-Side Liquidity (SSL) resting below old lows
        target_type = "Sell-Side Liquidity (SSL)"
        target_desc = "below swing lows"
        emoji = "📉"
        
        # Find all swing lows below current price
        valid_targets = recent_df[(recent_df['is_swing_low']) & (recent_df['low'] < current_price)]['low']
        
        if not valid_targets.empty:
            nearest_tp = float(valid_targets.max())
            major_tp = float(valid_targets.min())
        else:
            # Fallback if no prominent swings found
            lows_below = recent_df['low'][recent_df['low'] < current_price]
            nearest_tp = float(lows_below.max()) if not lows_below.empty else current_price * 0.99
            major_tp = float(lows_below.min()) if not lows_below.empty else current_price * 0.98

    # Calculations
    nearest_dist = abs(nearest_tp - current_price)
    nearest_pct = (nearest_dist / current_price) * 100
    
    major_dist = abs(major_tp - current_price)
    major_pct = (major_dist / current_price) * 100

    # Display Output
    print(f"\n💧 {emoji} LIQUIDITY POOL TARGETS ({bias} BIAS)")
    print("   " + "─" * 56)
    print(f"   Pool Type : {target_type}")
    print(f"   Location  : Resting {target_desc}")
    print(f"   Lookback  : {lookback} bars")
    print(f"   Current   : {current_price:.2f}")
    print("   " + "─" * 56)
    print(f"   🎯 TP 1 (Nearest) : {nearest_tp:.2f}")
    print(f"      Distance       : {nearest_dist:.2f} ({nearest_pct:.2f}%)")
    print(f"   🚀 TP 2 (Major)   : {major_tp:.2f}")
    print(f"      Distance       : {major_dist:.2f} ({major_pct:.2f}%)")
    print("   " + "─" * 56)
    
    # Contextual Advice
    if nearest_pct < 0.1:
        print("   ⚠️  CAUTION: Liquidity pool is very close. Risk/Reward may be poor.")
    else:
        print("   ✅ SETUP: Valid distance for target placement.")

    return {
        'valid': True,
        'type': target_type,
        'current_price': current_price,
        'tp_nearest': nearest_tp,
        'tp_nearest_dist': nearest_dist,
        'tp_nearest_pct': nearest_pct,
        'tp_major': major_tp,
        'tp_major_dist': major_dist,
        'tp_major_pct': major_pct
    }

# ───────────────────────────────────────────────────────────────────────────────
# RUN ANALYSIS
# ───────────────────────────────────────────────────────────────────────────────

print("═" * 70)
print("💧 LIQUIDITY POOL DETECTOR v2.0")
print("═" * 70)

# Determine bias from MSS or fallback
target_bias = 'BULLISH' if 'BULLISH' in mss.get('structure', '') else 'BEARISH'

# Execute the engine
lp_results = find_liquidity_pools(df, bias=target_bias, lookback=50, swing_window=3)

print("═" * 70)

# ── Export variables ───────────────────────────────────────────────────────────
lp = lp_results
print(f"\n✅ Exports: lp (dict with tp_nearest, tp_major, distances, etc.)")

══════════════════════════════════════════════════════════════════════
💧 LIQUIDITY POOL DETECTOR v2.0
══════════════════════════════════════════════════════════════════════

💧 📉 LIQUIDITY POOL TARGETS (BEARISH BIAS)
   ────────────────────────────────────────────────────────
   Pool Type : Sell-Side Liquidity (SSL)
   Location  : Resting below swing lows
   Lookback  : 50 bars
   Current   : 5096.97
   ────────────────────────────────────────────────────────
   🎯 TP 1 (Nearest) : 5093.43
      Distance       : 3.54 (0.07%)
   🚀 TP 2 (Major)   : 5088.42
      Distance       : 8.55 (0.17%)
   ────────────────────────────────────────────────────────
   ⚠️  CAUTION: Liquidity pool is very close. Risk/Reward may be poor.
══════════════════════════════════════════════════════════════════════

✅ Exports: lp (dict with tp_nearest, tp_major, distances, etc.)


---
## 🧠 Concept 9 — Full ICT Confluence Engine
Combines ALL concepts into a single signal. Requires **2.0+ score** to generate BUY/SELL.

| Signal | Score Weight |
|--------|-------------|
| Liquidity Sweep (aligned) | +1.0 |
| Fair Value Gap  (aligned) | +1.0 |
| Order Block     (aligned) | +1.0 |
| Dealing Range Zone        | +0.5 |
| OTE Zone active           | +0.5 |
| Silver Bullet Window      | +0.3 |

In [31]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11 — FULL CONFLUENCE ENGINE v2.0 (THE MASTER BRAIN)
# ═══════════════════════════════════════════════════════════════════════════════
# Features: Aggregates all ICT concepts, scores confluences, evaluates Risk:Reward,
#           and generates actionable, institutional-grade trade signals.

import pandas as pd
import numpy as np
from datetime import datetime

# ───────────────────────────────────────────────────────────────────────────────
# SAFETY & MOCK FALLBACKS (In case previous cells weren't run)
# ───────────────────────────────────────────────────────────────────────────────

try:
    _ = df.shape
except NameError:
    print("⚠️  Creating sample data for Confluence Engine...")
    dates = pd.date_range(end=pd.Timestamp.now(), periods=200, freq='15min')
    df = pd.DataFrame({'close': np.random.normal(5100, 20, 200)}, index=dates)

# Mocking missing detector functions to ensure the engine always runs smoothly
if 'detect_market_structure' not in globals():
    detect_market_structure = lambda d: {'structure': 'BULLISH', 'strength': 0.8}
if 'detect_liquidity_sweep' not in globals():
    detect_liquidity_sweep = lambda d: {'detected': True, 'type': 'BELOW_LOW', 'strength': 0.7, 'swept_level': float(d['close'].iloc[-1])*0.99}
if 'detect_fair_value_gap' not in globals():
    detect_fair_value_gap = lambda d: {'detected': True, 'type': 'BULLISH', 'strength': 0.85, 'high': float(d['close'].iloc[-1])*1.01, 'low': float(d['close'].iloc[-1])*0.99}
if 'analyze_dealing_range' not in globals():
    analyze_dealing_range = lambda d: {'zone': 'DISCOUNT'}
if 'check_silver_bullet_time' not in globals():
    check_silver_bullet_time = lambda: {'active': True, 'window': 'AM Session'}
    
# Link to upgraded components from Cells 7, 8, and 10
has_ob_engine = 'run_order_block_analysis' in globals()
has_ote_engine = 'check_ote_levels' in globals()
has_lp_engine = 'find_liquidity_pools' in globals()

# ───────────────────────────────────────────────────────────────────────────────
# CORE CONFLUENCE ENGINE
# ───────────────────────────────────────────────────────────────────────────────

def run_ict_confluence_engine(df: pd.DataFrame):
    print("═" * 70)
    print(" 🧠 ICT FULL CONFLUENCE ENGINE v2.0")
    print("═" * 70)
    
    df2 = df.copy()
    current_price = float(df2['close'].iloc[-1])
    
    # 1. Base Market Structure & Bias
    mss_r = detect_market_structure(df2)
    bias = 'BULLISH' if 'BULLISH' in mss_r.get('structure', '') else ('BEARISH' if 'BEARISH' in mss_r.get('structure', '') else 'NEUTRAL')

    if bias == 'NEUTRAL':
        print("\n⚪ SIGNAL: HOLD — No clear market structure shift detected.")
        return {'action': 'HOLD', 'bias': 'NEUTRAL', 'confidence': 0.0, 'score': 0.0, 'sl': None, 'tp': None, 'signals':[]}

    # 2. Run Sub-Engines
    sweep_r = detect_liquidity_sweep(df2)
    fvg_r   = detect_fair_value_gap(df2)
    deal_r  = analyze_dealing_range(df2)
    sb_r    = check_silver_bullet_time()
    
    # Execute upgraded engines if available, else mock (Fixed parenthesis location)
    ob_r = run_order_block_analysis(df2, bias=bias) if has_ob_engine else {'primary_ob': type('MockOB', (object,), {'strength': 0.8, 'low': current_price*0.99, 'high': current_price*1.01, 'type': bias})()}
    ote_r = check_ote_levels(df2, target_zone=fvg_r, zone_type="FVG", bias=bias) if has_ote_engine else {'valid': True}
    lp_r = find_liquidity_pools(df2, bias=bias, lookback=50) if has_lp_engine else {'tp_nearest': current_price*1.02 if bias=='BULLISH' else current_price*0.98, 'valid': True}

    # 3. Confluence Scoring Matrix
    score, max_score = 0.0, 4.5
    signals, strengths = [],[]
    sl_candidates =[]

    # Sweep Validation
    if sweep_r.get('detected'):
        if (bias == 'BULLISH' and sweep_r.get('type') == 'BELOW_LOW') or (bias == 'BEARISH' and sweep_r.get('type') == 'ABOVE_HIGH'):
            score += 1.0
            signals.append("✅ Liquidity Sweep Align")
            strengths.append(sweep_r.get('strength', 0.5))
            sl_candidates.append(sweep_r.get('swept_level', current_price))

    # FVG Validation
    if fvg_r.get('detected') and fvg_r.get('type') == bias:
        score += 1.0
        signals.append(f"✅ FVG Foundation ({fvg_r.get('strength', 0)*100:.0f}% str)")
        strengths.append(fvg_r.get('strength', 0.5))
        sl_candidates.append(fvg_r['low'] if bias == 'BULLISH' else fvg_r['high'])

    # Order Block Validation
    primary_ob = ob_r.get('primary_ob') if type(ob_r) == dict else None
    if primary_ob and getattr(primary_ob, 'type', None) == bias:
        ob_str = getattr(primary_ob, 'strength', 0.5)
        score += 1.0
        signals.append(f"✅ Institutional Order Block ({ob_str*100:.0f}% str)")
        strengths.append(ob_str)
        sl_candidates.append(primary_ob.low if bias == 'BULLISH' else primary_ob.high)

    # Dealing Range Validation
    if deal_r.get('zone') == ('DISCOUNT' if bias == 'BULLISH' else 'PREMIUM'):
        score += 0.5
        signals.append(f"✅ {deal_r.get('zone')} Pricing")

    # OTE Validation
    if ote_r.get('valid'):
        score += 0.5
        signals.append("✅ Deep OTE Pullback (62-79%)")

    # Time Validation
    if sb_r.get('active'):
        score += 0.5
        signals.append(f"✅ Silver Bullet Time ({sb_r.get('window')})")

    # 4. Math & Risk Calculations
    avg_str = sum(strengths) / len(strengths) if strengths else 0
    confidence = min(avg_str + (score / max_score) * 0.5, 1.0)
    
    # Action Trigger (Require at least 2.5 points for a valid setup)
    action = ('BUY' if bias == 'BULLISH' else 'SELL') if score >= 2.5 else 'HOLD'

    # Print Report
    print(f"\n   Bias       : {bias}")
    print(f"   Score      : {score:.1f} / {max_score:.1f}")
    print(f"   Confidence : {confidence:.0%}")
    print(f"\n   Confluences:")
    if not signals:
        print("      ❌ None detected")
    for s in signals: 
        print(f"      {s}")
    print(f"{'─'*70}")

    # 5. Trade Execution Logic
    sl = None
    tp = None
    
    if action != 'HOLD':
        # Smart Stop Loss (Safest invalidation point)
        if bias == 'BULLISH':
            base_sl = min(sl_candidates) if sl_candidates else current_price * 0.99
            sl = base_sl * 0.9995  # Breathing room
        else:
            base_sl = max(sl_candidates) if sl_candidates else current_price * 1.01
            sl = base_sl * 1.0005  # Breathing room
            
        # Target
        tp = lp_r.get('tp_nearest', current_price * (1.02 if action == 'BUY' else 0.98))
        
        # R:R Calculation
        risk = abs(current_price - sl)
        reward = abs(tp - current_price)
        rr_ratio = reward / risk if risk > 0 else 0
        
        rr_emoji = "⚠️" if rr_ratio < 1.5 else "✅"

        print(f"  🏹 SIGNAL   : {action} LIMIT / MKT")
        print(f"  💰 Entry    : {current_price:.2f}")
        print(f"  🛑 Stop Loss: {sl:.2f} (Invalidation Point)")
        print(f"  🎯 Target   : {tp:.2f} (Liquidity Pool)")
        print(f"  ⚖️ R:R Ratio : {rr_ratio:.2f}x {rr_emoji}")
        
        if rr_ratio < 1.5:
            print("     -> NOTE: R:R is below 1.5. Consider a tighter entry or skipping.")

    else:
        print(f"  ⏳ SIGNAL   : HOLD (Need score >= 2.5, currently {score:.1f})")
        print("     -> NOTE: Wait for more confluences to align.")
        
    print("═" * 70)
    return {
        'action': action, 'bias': bias, 'confidence': confidence, 
        'score': score, 'signals': signals, 'sl': sl, 'tp': tp
    }

# ───────────────────────────────────────────────────────────────────────────────
# EXECUTE THE ENGINE
# ───────────────────────────────────────────────────────────────────────────────

master_signal = run_ict_confluence_engine(df)

# Export variable
print("\n✅ Exports: master_signal (dict with action, sl, tp, confidence, etc.)")

══════════════════════════════════════════════════════════════════════
 🧠 ICT FULL CONFLUENCE ENGINE v2.0
══════════════════════════════════════════════════════════════════════
⚪ Market Structure : NEUTRAL
   Prev High : 5104.10  |  Curr High : 5104.10
   Prev Low  : 5092.36  |  Curr Low  : 5095.20

⚪ SIGNAL: HOLD — No clear market structure shift detected.

✅ Exports: master_signal (dict with action, sl, tp, confidence, etc.)


---
## 📈 Concept 10 — Walk-Forward Backtest
Tests the ICT confluence engine on historical bars.  
For each signal, checks price after **5 bars** to determine win/loss.

In [32]:
# ═══════════════════════════════════════════════════
# CELL 12 — WALK-FORWARD BACKTEST
# ═══════════════════════════════════════════════════
def backtest_ict(df, start_bar=100, min_score=1.5, forward_bars=5):
    print(f"📈 Walk-Forward Backtest | Min Score: {min_score} | Forward: {forward_bars} bars")
    print("─" * 55)
    results = []
    for i in range(start_bar, len(df) - forward_bars):
        window = df.iloc[:i+1].copy()
        try:
            mss_r   = detect_market_structure(window)
            bias    = 'BULLISH' if 'BULLISH' in mss_r['structure'] else ('BEARISH' if 'BEARISH' in mss_r['structure'] else 'NEUTRAL')
            if bias == 'NEUTRAL': continue
            sweep_r = detect_liquidity_sweep(window)
            fvg_r   = detect_fair_value_gap(window)
            score   = 0
            if sweep_r['detected']:
                if (bias=='BULLISH' and sweep_r['type']=='BELOW_LOW') or (bias=='BEARISH' and sweep_r['type']=='ABOVE_HIGH'):
                    score += 1
            if fvg_r.get('detected') and fvg_r.get('type') == bias:
                score += 1
            if score >= min_score:
                entry  = float(window['close'].iloc[-1])
                future = df['close'].iloc[i+1:i+1+forward_bars]
                action = 'BUY' if bias == 'BULLISH' else 'SELL'
                exit_p = float(future.iloc[-1])
                pnl    = (exit_p - entry) if action == 'BUY' else (entry - exit_p)
                results.append({'bar': i, 'action': action, 'entry': entry, 'exit': exit_p, 'pnl': pnl, 'win': pnl > 0})
        except: continue

    if not results:
        print("❌ No signals generated")
        return []

    total     = len(results)
    wins      = sum(1 for r in results if r['win'])
    total_pnl = sum(r['pnl'] for r in results)
    win_rate  = wins / total * 100
    avg_win   = sum(r['pnl'] for r in results if r['win']) / max(wins, 1)
    avg_loss  = sum(r['pnl'] for r in results if not r['win']) / max(total-wins, 1)

    print(f"  Total Signals : {total}")
    print(f"  Wins          : {wins}  ({win_rate:.1f}%)")
    print(f"  Losses        : {total - wins}")
    print(f"  Total PnL     : {total_pnl:+.2f} pts")
    print(f"  Avg Win       : {avg_win:+.2f} pts")
    print(f"  Avg Loss      : {avg_loss:+.2f} pts")
    if avg_loss != 0:
        rr = abs(avg_win / avg_loss)
        print(f"  R:R Ratio     : 1:{rr:.2f}")
    print("─" * 55)
    print("  Last 5 Trades:")
    for r in results[-5:]:
        e = '✅' if r['win'] else '❌'
        print(f"  {e} {r['action']:4s} {r['entry']:.2f} → {r['exit']:.2f}  ({r['pnl']:+.2f} pts)")
    return results

results = backtest_ict(df, start_bar=100, min_score=1.5, forward_bars=5)

📈 Walk-Forward Backtest | Min Score: 1.5 | Forward: 5 bars
───────────────────────────────────────────────────────
⚪ Market Structure : NEUTRAL
   Prev High : 5101.86  |  Curr High : 5100.90
   Prev Low  : 5087.91  |  Curr Low  : 5096.09
⚪ Market Structure : NEUTRAL
   Prev High : 5101.86  |  Curr High : 5099.65
   Prev Low  : 5085.51  |  Curr Low  : 5085.51
⚪ Market Structure : NEUTRAL
   Prev High : 5100.90  |  Curr High : 5092.01
   Prev Low  : 5084.65  |  Curr Low  : 5084.65
⚪ Market Structure : NEUTRAL
   Prev High : 5099.65  |  Curr High : 5086.82
   Prev Low  : 5074.08  |  Curr Low  : 5074.08
⚪ Market Structure : NEUTRAL
   Prev High : 5092.01  |  Curr High : 5091.27
   Prev Low  : 5074.08  |  Curr Low  : 5082.70
⚪ Market Structure : NEUTRAL
   Prev High : 5091.27  |  Curr High : 5090.91
   Prev Low  : 5074.08  |  Curr Low  : 5083.79
⚪ Market Structure : NEUTRAL
   Prev High : 5091.27  |  Curr High : 5090.13
   Prev Low  : 5078.12  |  Curr Low  : 5078.12
⚪ Market Structure : NEU

---
## ✅ Notebook Complete

### Summary
| Concept | Purpose |
|---------|--------|
| MSS/BOS/ChoCH | Determine trend direction |
| Liquidity Sweep | Detect stop hunts for reversal |
| FVG | Find imbalance zones for entry |
| Dealing Range | Confirm buy/sell zone alignment |
| Order Block | Find institutional entry zones |
| OTE | Fibonacci precision entry |
| Silver Bullet | Time-based probability boost |
| Liquidity Pool | Set accurate take profit |

> **Next Step:** Modify parameters in each cell and observe how signals change. Try different symbols and timeframes!

Built by **Next Level Brain** 🧠